<a href="https://colab.research.google.com/github/TK-Problem/random-experiments/blob/main/001_bayesian_linear_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from datetime import date, timedelta

## Configuration

Tickers, date range, lag, and split settings used throughout the experiment.

In [ ]:
N_DAYS = 60   # Total trading days to keep
N_TEST = 5    # Most recent days reserved for testing

TARGET = "SPY"
PREDICTORS = [
    "QQQ",   # Nasdaq 100
    "IWM",   # Russell 2000 (small cap)
    "GLD",   # Gold
    "TLT",   # Long-term Treasuries
    "VIX",   # Volatility index (^VIX in yfinance)
    "XLE",   # Energy sector
    "XLF",   # Financials sector
    "EEM",   # Emerging markets
]

# Request ~1.5x calendar days to guarantee N_DAYS trading days after weekends/holidays
END_DATE = date.today().strftime("%Y-%m-%d")
START_DATE = (date.today() - timedelta(days=int(N_DAYS * 1.5))).strftime("%Y-%m-%d")
LAG = 1   # Days to lag predictors (1 = previous day)
RANDOM_SEED = 42

## Load Data

Download daily closing prices from Yahoo Finance and compute log returns.

In [ ]:
print("Downloading price data from Yahoo Finance...")
tickers = [TARGET] + PREDICTORS
# ^VIX needs special handling
yf_tickers = [t if t != "VIX" else "^VIX" for t in tickers]

raw = yf.download(yf_tickers, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]

# Rename ^VIX back to VIX
raw.columns = [c.replace("^", "") for c in raw.columns]

# Compute daily log returns
returns = np.log(raw / raw.shift(1)).dropna()
print(f"Downloaded {len(returns)} trading days of data.\n")

In [ ]:
returns.head()

## Feature Engineering

Lag predictor returns by `LAG` days so features only use information available before the target date.

In [ ]:
# Target: SPY return on day t
y_series = returns[TARGET].copy()

# Features: each predictor's return on day t-LAG
X_df = returns[PREDICTORS].shift(LAG)

# Align and drop NaNs
df = pd.concat([y_series.rename("target"), X_df], axis=1).dropna()

y = df["target"].values
X_raw = df[PREDICTORS].values

print(f"Feature matrix shape: {X_raw.shape}")
print(f"Target vector shape:  {y.shape}\n")

## Train / Test Split

Standardize features and split chronologically (no shuffle) to avoid look-ahead bias.

## Single-Beta Bayesian Models

Fit one PyMC model per predictor on the training set to obtain individual beta estimates.

In [ ]:
single_traces = {}

print("Fitting single-beta models...\n")
for i, predictor in enumerate(PREDICTORS):
    x_tr = X_train[:, i]

    with pm.Model():
        alpha = pm.Normal("alpha", mu=0, sigma=0.01)
        beta  = pm.Normal("beta",  mu=0, sigma=0.05)
        sigma = pm.HalfNormal("sigma", sigma=0.01)
        mu    = alpha + beta * x_tr
        _     = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

        trace = pm.sample(
            draws=1000, tune=500, chains=2,
            target_accept=0.9, random_seed=RANDOM_SEED,
            progressbar=False,
        )

    single_traces[predictor] = trace
    b_mean = float(trace.posterior["beta"].values.mean())
    b_std  = float(trace.posterior["beta"].values.std())
    rhat   = float(az.summary(trace, var_names=["beta"])["r_hat"].values[0])
    print(f"  {predictor:6s}  beta={b_mean:+.5f} ± {b_std:.5f}  r_hat={rhat:.3f}")

print("\nSingle-beta sampling complete.")

## Single-Beta Evaluation

Out-of-sample metrics for each individual predictor model.

In [ ]:
print("--- Single-Beta Out-of-Sample Metrics ---\n")
single_metrics = []

for i, predictor in enumerate(PREDICTORS):
    trace    = single_traces[predictor]
    x_te     = X_test[:, i]
    post_a   = trace.posterior["alpha"].values.flatten()
    post_b   = trace.posterior["beta"].values.flatten()

    mu_pred     = post_a[:, None] + post_b[:, None] * x_te[None, :]
    y_pred_mean = mu_pred.mean(axis=0)

    ss_res = np.sum((y_test - y_pred_mean) ** 2)
    ss_tot = np.sum((y_test - y_test.mean()) ** 2)
    r2   = 1 - ss_res / ss_tot
    rmse = np.sqrt(np.mean((y_test - y_pred_mean) ** 2))
    corr = np.corrcoef(y_test, y_pred_mean)[0, 1]

    single_metrics.append({"ETF": predictor, "R²": r2, "RMSE": rmse, "Corr": corr})
    print(f"  {predictor:6s}  R²={r2:+.4f}  RMSE={rmse:.6f}  Corr={corr:+.4f}")

single_metrics_df = pd.DataFrame(single_metrics).set_index("ETF")
print(f"\nBest single predictor by R²: {single_metrics_df['R²'].idxmax()}")

## Multi-Variable Bayesian Linear Regression

Now that individual predictors have been assessed, fit a joint model with all predictors simultaneously.

In [ ]:
n_features = X_train.shape[1]

with pm.Model() as model:
    alpha  = pm.Normal("alpha", mu=0, sigma=0.01)
    betas  = pm.Normal("betas", mu=0, sigma=0.05, shape=n_features)
    sigma  = pm.HalfNormal("sigma", sigma=0.01)
    mu     = alpha + pm.math.dot(X_train, betas)
    _      = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_train)

    print("Sampling posterior (this may take a minute)...")
    trace = pm.sample(
        draws=2000, tune=1000, chains=4,
        target_accept=0.9, random_seed=RANDOM_SEED,
        progressbar=True,
    )

print("\nSampling complete!")
summary = az.summary(trace, var_names=["alpha", "betas", "sigma"], round_to=6)
summary.index = ["alpha"] + [f"beta_{p}" for p in PREDICTORS] + ["sigma"]
print("\n--- Posterior Summary ---")
print(summary)

## In-Sample Diagnostics

Trace plots and forest plot to check sampler convergence and posterior coefficient estimates.

In [ ]:
az.plot_trace(trace, var_names=["alpha", "betas", "sigma"], compact=True)
plt.suptitle("Trace Plots", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("trace_plots.png", dpi=100, bbox_inches="tight")
plt.show()

az.plot_forest(trace, var_names=["betas"],
               combined=True, hdi_prob=0.94, r_hat=True)
plt.axvline(0, color="red", linestyle="--", alpha=0.6)
plt.yticks(ticks=range(n_features), labels=PREDICTORS[::-1])
plt.title("Beta Coefficients — 94% HDI")
plt.tight_layout()
plt.savefig("forest_plot.png", dpi=100, bbox_inches="tight")
plt.show()

## Out-of-Sample Prediction

Compare multi-variable model performance against the single-beta baselines.

In [ ]:
post_alpha = trace.posterior["alpha"].values.flatten()
post_betas = trace.posterior["betas"].values.reshape(-1, n_features)
post_sigma = trace.posterior["sigma"].values.flatten()

mu_pred     = post_alpha[:, None] + post_betas @ X_test.T
y_pred_mean = mu_pred.mean(axis=0)
y_pred_hdi  = az.hdi(mu_pred, hdi_prob=0.94)

ss_res = np.sum((y_test - y_pred_mean) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2   = 1 - ss_res / ss_tot
rmse = np.sqrt(np.mean((y_test - y_pred_mean) ** 2))
corr = np.corrcoef(y_test, y_pred_mean)[0, 1]

print("--- Out-of-Sample Metrics Comparison ---\n")
print(f"  {'Model':<20}  {'R²':>8}  {'RMSE':>10}  {'Corr':>8}")
print(f"  {'-'*50}")
for _, row in single_metrics_df.iterrows():
    print(f"  {row.name:<20}  {row['R²']:>+8.4f}  {row['RMSE']:>10.6f}  {row['Corr']:>+8.4f}")
print(f"  {'-'*50}")
print(f"  {'Multi-variable':<20}  {r2:>+8.4f}  {rmse:>10.6f}  {corr:>+8.4f}")

## Visualize Predictions

Plot actual vs. predicted SPY log returns and residuals over the test period.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(dates_test, y_test,      label="Actual SPY return",          alpha=0.7, linewidth=0.8)
axes[0].plot(dates_test, y_pred_mean, label="Predicted (posterior mean)",  alpha=0.9, linewidth=0.8)
axes[0].fill_between(dates_test,
                     y_pred_hdi[:, 0], y_pred_hdi[:, 1],
                     alpha=0.2, label="94% HDI")
axes[0].axhline(0, color="black", linewidth=0.5, linestyle="--")
axes[0].set_title(f"SPY Daily Return — Multi-variable Model  |  R²={r2:.4f}  RMSE={rmse:.6f}")
axes[0].legend(fontsize=9)
axes[0].set_ylabel("Log Return")

residuals = y_test - y_pred_mean
axes[1].bar(dates_test, residuals, width=1, alpha=0.6, color="steelblue", label="Residuals")
axes[1].axhline(0, color="red", linewidth=0.8)
axes[1].set_title("Residuals")
axes[1].set_ylabel("Error")
axes[1].set_xlabel("Date")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("predictions.png", dpi=100, bbox_inches="tight")
plt.show()

## Coefficient Importance

Rank predictors by the Bayesian signal-to-noise ratio (|mean| / std) and plot posterior beta estimates.

In [ ]:
beta_means = post_betas.mean(axis=0)
beta_stds  = post_betas.std(axis=0)

coef_df = pd.DataFrame({
    "ETF":       PREDICTORS,
    "beta_mean": beta_means,
    "beta_std":  beta_stds,
    "signal":    np.abs(beta_means) / beta_stds,
}).sort_values("signal", ascending=False)

print("--- Coefficient Summary (sorted by |mean/std|) ---")
print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v > 0 else "#2980b9" for v in coef_df["beta_mean"]]
ax.barh(coef_df["ETF"], coef_df["beta_mean"], xerr=coef_df["beta_std"],
        color=colors, capsize=4, alpha=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Posterior Beta Coefficients (mean ± 1 std)")
ax.set_xlabel("Coefficient value")
plt.tight_layout()
plt.savefig("coefficients.png", dpi=100, bbox_inches="tight")
plt.show()